<a href="https://colab.research.google.com/github/DtheFather/Emperical-Evaluation-of-XAI-on-Edge-devices-in-Medical-Diagnostic-Vision/blob/main/EfficientNet_B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q captum lime medmnist psutil scipy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import time
import csv
import psutil
import zipfile
import shutil
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from captum.attr import LayerGradCam, IntegratedGradients
from lime import lime_image
from scipy.integrate import simpson

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class CustomMedMNISTDataset(Dataset):
    def __init__(self, npz_path, split='train', transform=None, extract_dir='/content/tmp_mmap'):
        self.transform = transform
        os.makedirs(extract_dir, exist_ok=True)

        base_name = os.path.basename(npz_path).replace('.npz', '')
        self.img_path = os.path.join(extract_dir, f"{base_name}_{split}_images.npy")
        self.lbl_path = os.path.join(extract_dir, f"{base_name}_{split}_labels.npy")

        if not (os.path.exists(self.img_path) and os.path.exists(self.lbl_path)):
            print(f"Streaming {split} arrays to disk for {base_name}...")
            with zipfile.ZipFile(npz_path, 'r') as zip_ref:
                with zip_ref.open(f'{split}_images.npy') as src, open(self.img_path, 'wb') as tgt:
                    shutil.copyfileobj(src, tgt)
                with zip_ref.open(f'{split}_labels.npy') as src, open(self.lbl_path, 'wb') as tgt:
                    shutil.copyfileobj(src, tgt)

        self.images = np.load(self.img_path, mmap_mode='r')
        self.labels = np.load(self.lbl_path, mmap_mode='r')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        labels = self.labels[idx].astype(np.float32)

        img = Image.fromarray(img_array)
        if self.transform:
            img = self.transform(img)

        return img, labels

def prepare_dataloaders(npz_file_name, data_dir, batch_size=64, num_bench_samples=100):
    npz_path = os.path.join(data_dir, npz_file_name)
    data_transform = transforms.Compose([
        transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(CustomMedMNISTDataset(npz_path, 'train', data_transform), batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(CustomMedMNISTDataset(npz_path, 'val', data_transform), batch_size=batch_size, shuffle=False, num_workers=2)

    test_dataset = CustomMedMNISTDataset(npz_path, 'test', data_transform)
    bench_loader = DataLoader(Subset(test_dataset, list(range(num_bench_samples))), batch_size=1, shuffle=False)

    return train_loader, val_loader, bench_loader


In [ ]:
def build_efficientnet_b0(num_classes):
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def train_model(model, train_loader, val_loader, task_type, device, epochs=10):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss() if task_type == 'multi-label, binary-class' else nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            targets = targets.float() if task_type == 'multi-label, binary-class' else targets.squeeze().long()

            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)

        print(f"Epoch {epoch+1}/{epochs} - Training Loss: {running_loss / len(train_loader.dataset):.4f}")

    return model

In [ ]:
class EfficientNetXAITelemetryEngine:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.model.eval()
        self.device = device

        self.grad_cam = LayerGradCam(self.model, self.model.features[-1])
        self.ig = IntegratedGradients(self.model)
        self.lime_explainer = lime_image.LimeImageExplainer()

    def get_process_ram_mb(self):
        return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

    def extract_gradcam(self, input_tensor, target_class):
        start_ram = self.get_process_ram_mb()
        start_time = time.perf_counter()

        attr = self.grad_cam.attribute(input_tensor.to(self.device), target=target_class)

        elapsed_ms = (time.perf_counter() - start_time) * 1000
        peak_ram = self.get_process_ram_mb() - start_ram
        return attr.detach().cpu(), elapsed_ms, peak_ram

    def extract_integrated_gradients(self, input_tensor, target_class, steps=50):
        start_ram = self.get_process_ram_mb()
        start_time = time.perf_counter()

        attr = self.ig.attribute(input_tensor.to(self.device), target=target_class, n_steps=steps)

        elapsed_ms = (time.perf_counter() - start_time) * 1000
        peak_ram = self.get_process_ram_mb() - start_ram
        return attr.detach().cpu(), elapsed_ms, peak_ram

    def extract_lime(self, input_tensor, target_class, num_samples=500):
        start_ram = self.get_process_ram_mb()
        start_time = time.perf_counter()

        img_np = input_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

        def predict_fn(images):
            batch = torch.tensor(images).permute(0, 3, 1, 2).float().to(self.device)
            with torch.no_grad():
                return self.model(batch).cpu().numpy()

        explanation = self.lime_explainer.explain_instance(
            img_np, predict_fn, labels=(target_class,), num_samples=num_samples, top_labels=1
        )

        elapsed_ms = (time.perf_counter() - start_time) * 1000
        peak_ram = self.get_process_ram_mb() - start_ram
        return explanation, elapsed_ms, peak_ram

def evaluate_insertion_deletion(model, input_tensor, attribution_map, target_class, steps=10, device='cuda'):
    model.eval()
    if attribution_map.shape[-2:] != input_tensor.shape[-2:]:
        attribution_map = F.interpolate(attribution_map, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
    if attribution_map.shape[1] == 1 and input_tensor.shape[1] == 3:
        attribution_map = attribution_map.expand_as(input_tensor)

    flat_attr = attribution_map.flatten()
    sorted_indices = torch.argsort(flat_attr, descending=True)
    step_size = flat_attr.numel() // steps

    input_tensor, baseline = input_tensor.to(device), torch.zeros_like(input_tensor).to(device)
    deletion_scores, insertion_scores = [], []

    with torch.no_grad():
        deletion_scores.append(torch.softmax(model(input_tensor), dim=1)[0, target_class].item())
        insertion_scores.append(torch.softmax(model(baseline), dim=1)[0, target_class].item())

        for s in range(1, steps + 1):
            mask_idx = sorted_indices[: s * step_size]

            del_img = input_tensor.clone()
            del_img.view(-1)[mask_idx] = 0.0
            deletion_scores.append(torch.softmax(model(del_img), dim=1)[0, target_class].item())

            ins_img = baseline.clone()
            ins_img.view(-1)[mask_idx] = input_tensor.view(-1)[mask_idx]
            insertion_scores.append(torch.softmax(model(ins_img), dim=1)[0, target_class].item())

    x_axis = np.linspace(0, 1, len(deletion_scores))
    return simpson(deletion_scores, x=x_axis), simpson(insertion_scores, x=x_axis)

In [ ]:
def log_to_csv(csv_path, row_data):
    with open(csv_path, mode='a', newline='') as file:
        csv.writer(file).writerow(row_data)

def run_hardware_benchmark(xai_engine, bench_loader, csv_path):
    with open(csv_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Image_ID", "True_Label", "Algorithm", "Latency_ms", "RAM_Spike_MB", "Deletion_AUC", "Insertion_AUC"])

    for idx, (input_tensor, targets) in enumerate(bench_loader):
        target_class = torch.argmax(targets[0]).item() if targets.ndim > 1 and targets.shape[1] > 1 else int(targets[0].item())
        print(f"Processing Image {idx+1}/100 [True Label: {target_class}]")

        gc_attr, gc_lat, gc_ram = xai_engine.extract_gradcam(input_tensor, target_class)
        gc_del, gc_ins = evaluate_insertion_deletion(xai_engine.model, input_tensor, gc_attr, target_class, device=xai_engine.device)
        log_to_csv(csv_path, [idx, target_class, "Grad-CAM", gc_lat, gc_ram, gc_del, gc_ins])

        ig_attr, ig_lat, ig_ram = xai_engine.extract_integrated_gradients(input_tensor, target_class)
        ig_del, ig_ins = evaluate_insertion_deletion(xai_engine.model, input_tensor, ig_attr, target_class, device=xai_engine.device)
        log_to_csv(csv_path, [idx, target_class, "Integrated_Gradients", ig_lat, ig_ram, ig_del, ig_ins])

        lime_explanation, lime_lat, lime_ram = xai_engine.extract_lime(input_tensor, target_class)
        lime_mask = torch.tensor(
            lime_explanation.get_image_and_mask(lime_explanation.top_labels[0], positive_only=False, num_features=10, hide_rest=False)[1]
        ).float().unsqueeze(0).unsqueeze(0)

        lime_del, lime_ins = evaluate_insertion_deletion(xai_engine.model, input_tensor, lime_mask, target_class, device=xai_engine.device)
        log_to_csv(csv_path, [idx, target_class, "LIME", lime_lat, lime_ram, lime_del, lime_ins])

DRIVE_DATA_DIR = '/content/drive/MyDrive/CAPSTONE/Datasets/MEDMNIST'
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_efficientnet_pipeline(dataset_name, task_type, num_classes):
    dataset_prefix = dataset_name.split('_')[0]
    weight_path = f"/content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_{dataset_prefix}.pth"
    csv_out_path = f"/content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/XAI_Benchmark/xai_benchmark_{dataset_prefix}.csv"

    os.makedirs(os.path.dirname(weight_path), exist_ok=True)
    os.makedirs(os.path.dirname(csv_out_path), exist_ok=True)

    print(f"\n==========================================")
    print(f"Starting Pipeline for {dataset_name}")
    print(f"==========================================")

    train_loader, val_loader, bench_loader = prepare_dataloaders(dataset_name, DRIVE_DATA_DIR)
    model = build_efficientnet_b0(num_classes=num_classes)

    if os.path.exists(weight_path):
        print(f"Weights found at {weight_path}. Loading frozen parameters...")
        model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
        trained_model = model.to(DEVICE)
    else:
        print(f"No weights found at {weight_path}. Starting training loop...")
        trained_model = train_model(model, train_loader, val_loader, task_type, DEVICE, EPOCHS)
        torch.save(trained_model.state_dict(), weight_path)
        print(f"EfficientNet-B0 Model saved to {weight_path}")

    print("Initializing EfficientNet-B0 XAI Telemetry Engine...")
    xai_engine = EfficientNetXAITelemetryEngine(model=trained_model, device=DEVICE)

    print(f"Running benchmarks. Logging to {csv_out_path}...")
    run_hardware_benchmark(xai_engine, bench_loader, csv_out_path)

    print(f"Pipeline Complete for {dataset_prefix}!")

    del model, trained_model, xai_engine, train_loader, val_loader, bench_loader
    torch.cuda.empty_cache()

In [ ]:
run_efficientnet_pipeline('pneumoniamnist_224.npz', task_type='binary-class', num_classes=2)


Starting Pipeline for pneumoniamnist_224.npz
Streaming train arrays to disk for pneumoniamnist_224...
Streaming val arrays to disk for pneumoniamnist_224...
Streaming test arrays to disk for pneumoniamnist_224...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 149MB/s]


Weights found at /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_pneumoniamnist.pth. Loading frozen parameters...
Initializing EfficientNet-B0 XAI Telemetry Engine...
Running benchmarks. Logging to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/XAI_Benchmark/xai_benchmark_pneumoniamnist.csv...
Processing Image 1/100 [True Label: 1]


  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 2/100 [True Label: 0]


  0%|          | 0/500 [00:00<?, ?it/s]

... (193 outputs left out)

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 100/100 [True Label: 1]


  0%|          | 0/500 [00:00<?, ?it/s]

Pipeline Complete for pneumoniamnist!


In [ ]:
run_efficientnet_pipeline('dermamnist_224.npz', task_type='multi-class', num_classes=7)


Starting Pipeline for dermamnist_224.npz
Streaming train arrays to disk for dermamnist_224...
Streaming val arrays to disk for dermamnist_224...
Streaming test arrays to disk for dermamnist_224...
No weights found at /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_dermamnist.pth. Starting training loop...
Epoch 1/10 - Training Loss: 0.6823
Epoch 2/10 - Training Loss: 0.4178
Epoch 3/10 - Training Loss: 0.2976
Epoch 4/10 - Training Loss: 0.2084
Epoch 5/10 - Training Loss: 0.1794
Epoch 6/10 - Training Loss: 0.1378
Epoch 7/10 - Training Loss: 0.1181
Epoch 8/10 - Training Loss: 0.0927
Epoch 9/10 - Training Loss: 0.0748
Epoch 10/10 - Training Loss: 0.0895
EfficientNet-B0 Model saved to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_dermamnist.pth
Initializing EfficientNet-B0 XAI Telemetry Engine...
Running benchmarks. Logging to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/XAI_Benchmark/xai_benchmark_dermamnist

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 2/100 [True Label: 3]


  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 3/100 [True Label: 4]


  0%|          | 0/500 [00:00<?, ?it/s]

... (191 outputs left out)

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 100/100 [True Label: 5]


  0%|          | 0/500 [00:00<?, ?it/s]

Pipeline Complete for dermamnist!


In [ ]:
run_efficientnet_pipeline('bloodmnist_224.npz', task_type='multi-class', num_classes=8)


Starting Pipeline for bloodmnist_224.npz
Streaming train arrays to disk for bloodmnist_224...
Streaming val arrays to disk for bloodmnist_224...
Streaming test arrays to disk for bloodmnist_224...
Weights found at /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_bloodmnist.pth. Loading frozen parameters...
Initializing EfficientNet-B0 XAI Telemetry Engine...
Running benchmarks. Logging to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/XAI_Benchmark/xai_benchmark_bloodmnist.csv...
Processing Image 1/100 [True Label: 3]


  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 2/100 [True Label: 0]


  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 3/100 [True Label: 1]


  0%|          | 0/500 [00:00<?, ?it/s]

... (191 outputs left out)

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 100/100 [True Label: 7]


  0%|          | 0/500 [00:00<?, ?it/s]

Pipeline Complete for bloodmnist!


In [ ]:
run_efficientnet_pipeline('chestmnist_224.npz', task_type='multi-label, binary-class', num_classes=14)


Starting Pipeline for chestmnist_224.npz
Streaming train arrays to disk for chestmnist_224...
Streaming val arrays to disk for chestmnist_224...
Streaming test arrays to disk for chestmnist_224...
No weights found at /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_chestmnist.pth. Starting training loop...
Epoch 1/10 - Training Loss: 0.1618
Epoch 2/10 - Training Loss: 0.1498
Epoch 3/10 - Training Loss: 0.1453
Epoch 4/10 - Training Loss: 0.1412
Epoch 5/10 - Training Loss: 0.1369
Epoch 6/10 - Training Loss: 0.1315
Epoch 7/10 - Training Loss: 0.1252
Epoch 8/10 - Training Loss: 0.1170
Epoch 9/10 - Training Loss: 0.1078
Epoch 10/10 - Training Loss: 0.0979
EfficientNet-B0 Model saved to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_chestmnist.pth
Initializing EfficientNet-B0 XAI Telemetry Engine...
Running benchmarks. Logging to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/XAI_Benchmark/xai_benchmark_chestmnist

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 2/100 [True Label: 0]


  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 3/100 [True Label: 3]


  0%|          | 0/500 [00:00<?, ?it/s]

... (191 outputs left out)

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 100/100 [True Label: 0]


  0%|          | 0/500 [00:00<?, ?it/s]

Pipeline Complete for chestmnist!


In [ ]:
run_efficientnet_pipeline('pathmnist_224.npz', task_type='multi-class', num_classes=9)


Starting Pipeline for pathmnist_224.npz
Streaming train arrays to disk for pathmnist_224...
Streaming val arrays to disk for pathmnist_224...
Streaming test arrays to disk for pathmnist_224...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 105MB/s]


No weights found at /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_pathmnist.pth. Starting training loop...
Epoch 1/10 - Training Loss: 0.0954
Epoch 2/10 - Training Loss: 0.0448
Epoch 3/10 - Training Loss: 0.0332
Epoch 4/10 - Training Loss: 0.0275
Epoch 5/10 - Training Loss: 0.0266
Epoch 6/10 - Training Loss: 0.0274
Epoch 7/10 - Training Loss: 0.0157
Epoch 8/10 - Training Loss: 0.0177
Epoch 9/10 - Training Loss: 0.0192
Epoch 10/10 - Training Loss: 0.0118
EfficientNet-B0 Model saved to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/ModelWeights/efficientnetb0_pathmnist.pth
Initializing EfficientNet-B0 XAI Telemetry Engine...
Running benchmarks. Logging to /content/drive/MyDrive/CAPSTONE/Models/EfficientNet-B0/XAI_Benchmark/xai_benchmark_pathmnist.csv...
Processing Image 1/100 [True Label: 8]


  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 2/100 [True Label: 4]


  0%|          | 0/500 [00:00<?, ?it/s]

... (193 outputs left out)

  0%|          | 0/500 [00:00<?, ?it/s]

Processing Image 100/100 [True Label: 1]


  0%|          | 0/500 [00:00<?, ?it/s]

Pipeline Complete for pathmnist!
